# Tema: COPY INTO y selección de ingesta

## Objetivos
Cargar archivos incrementalmente con SQL y elegir entre conectores e ingesta de ficheros.

## Conceptos importantes para el examen
COPY INTO recuerda ficheros cargados; no deduplica claves de negocio. Auto Loader escala descubrimiento incremental. Lakeflow Connect integra fuentes soportadas; JDBC/REST requieren gestionar extracción y credenciales.

**Dificultad:** Intermedio · **Tiempo estimado:** 65 min.



Ejecuta la preparación una vez; después avanza celda a celda. Las soluciones modifican datos: úsalas tras tu intento. Para volver al estado inicial, ejecuta de nuevo la preparación completa (crea otro schema). No uses «Run all» para estudiar.

- [ ] Completado
- [ ] Necesito repasar
- [ ] Dominado

## Preparación y datos ficticios
Se necesita un notebook Python en Databricks con Spark y Unity Catalog. Solo se crean objetos en el schema de prácticas mostrado.

In [ ]:
# Cada ejecución de esta celda crea un schema NUEVO y aislado.
# El catálogo debe existir y permitir USE CATALOG y CREATE SCHEMA.
# Si no puedes crear schemas, pide uno de prácticas exclusivo y cambia SCHEMA.
import re
import uuid
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window

dbutils.widgets.text("catalog", spark.sql("SELECT current_catalog()").first()[0])
CATALOG = dbutils.widgets.get("catalog")
RUN_ID = uuid.uuid4().hex[:10]
SCHEMA = "dea_17_" + RUN_ID
def ident(value):
    return "`" + value.replace("`", "``") + "`"
spark.sql(f"USE CATALOG {ident(CATALOG)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {ident(SCHEMA)}")
spark.sql(f"USE SCHEMA {ident(SCHEMA)}")
spark.sql("SET TIME ZONE 'UTC'")
print(f"Objetos de esta sesión: {CATALOG}.{SCHEMA}")
# No se borran automáticamente schemas, tablas ni checkpoints.


In [ ]:
import json
records = [{"event_id": i, "customer_id": i % 4, "amount": i * 10} for i in range(1, 13)]
# El directorio se crea en la siguiente celda; todavía no hay ninguna carga.

In [ ]:
# Requiere CREATE VOLUME en el schema; alternativa: usa un volumen autorizado.
spark.sql("CREATE VOLUME IF NOT EXISTS lab_files")
BASE = f"/Volumes/{CATALOG}/{SCHEMA}/lab_files"
dbutils.fs.mkdirs(BASE + "/landing")
CHECKPOINT = BASE + "/checkpoints/main"
print(BASE)
SOURCE = BASE + "/landing"
dbutils.fs.put(SOURCE + "/batch_01.json", "\n".join(json.dumps(r) for r in records), overwrite=False)

## PARTE 1 - EJEMPLOS GUIADOS

### 1. Crear destino y cargar

In [ ]:
spark.sql("CREATE TABLE copy_events (event_id BIGINT, customer_id BIGINT, amount BIGINT) USING DELTA")
def run_copy(force=False):
    return spark.sql(f"""COPY INTO copy_events FROM '{SOURCE}' FILEFORMAT = JSON
    COPY_OPTIONS ('force' = '{str(force).lower()}')""")
display(run_copy())

### 2. Repetir la carga

In [ ]:
display(run_copy())
assert spark.table("copy_events").count() == 12

### 3. Cliente REST ficticio sin red
Esta función simula páginas de una API; no ejecuta un conector real. En una integración real necesitarás autenticación, paginación, reintentos y cursor persistido.

In [ ]:
def fake_api(page):
    return records[(page-1)*4:page*4]
api_rows = [row for page in range(1,4) for row in fake_api(page)]
spark.createDataFrame(api_rows).write.format("delta").mode("overwrite").saveAsTable("rest_landing")

## PARTE 2 - EJERCICIOS
Resuelve todos antes de abrir las soluciones. Los ejercicios se realizan en orden y pueden usar resultados anteriores.

### EJERCICIO 1
Añade un fichero con eventos 13 y 14 y carga solo lo nuevo; verifica 14.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 2
Añade otro fichero que repita event_id 1 y observa el duplicado de negocio.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 3
Construye Silver con claves únicas y explica por qué force=true no arregla duplicados.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 4
Elige herramienta para 10 JSON diarios, millones de archivos, Salesforce soportado y una API interna sin conector. Guarda decisiones.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


### EJERCICIO 5
Procesa solo páginas 2–3 de fake_api y escribe una tabla de staging. Documenta cómo configurarías un conector real y una lectura JDBC.

In [ ]:
# ESCRIBE TU CÓDIGO AQUÍ


## PARTE 3 - PISTAS
**Pista 1:** No uses force.

**Pista 2:** La unidad de seguimiento es el fichero.

**Pista 3:** Para estas filas idénticas basta dropDuplicates.

**Pista 4:** Compara volumen, frecuencia, soporte de fuente y mantenimiento.

**Pista 5:** La simulación prepara la transformación; no prueba conectividad externa.

## PARTE 4 - SOLUCIONES
**Detente aquí si todavía estás practicando.** Referencias completas para comparar después de resolver. Puedes plegar esta sección en Databricks.

### Solución 1

In [ ]:
dbutils.fs.put(SOURCE + "/batch_02.json", "\n".join(json.dumps({"event_id":i,"customer_id":1,"amount":20}) for i in [13,14]), overwrite=False)
display(run_copy())
assert spark.table("copy_events").count() == 14

### Solución 2

In [ ]:
dbutils.fs.put(SOURCE + "/duplicate.json", json.dumps(records[0]), overwrite=False)
run_copy()
assert spark.table("copy_events").filter("event_id=1").count() == 2

### Solución 3

In [ ]:
spark.table("copy_events").dropDuplicates(["event_id"]).write.format("delta").mode("overwrite").saveAsTable("copy_silver")
assert spark.table("copy_silver").count() == 14
# force vuelve a cargar archivos ya procesados; puede aumentar duplicados.

### Solución 4

In [ ]:
display(spark.createDataFrame([
("10 JSON/día", "COPY INTO", "Carga SQL incremental sencilla"),
("Millones de archivos", "Auto Loader", "Descubrimiento incremental escalable"),
("Salesforce soportado", "Lakeflow Connect managed", "Conexión UC y pipeline administrado"),
("API interna", "REST + Lakeflow Jobs", "Implementar cursor, paginación y reintentos")], "source STRING, choice STRING, reason STRING"))

### Solución 5

In [ ]:
staging = spark.createDataFrame([row for p in [2,3] for row in fake_api(p)])
staging.write.format("delta").mode("overwrite").saveAsTable("api_staging")
assert spark.table("api_staging").count() == 8
# Ampliación con una fuente propia: UI Add data → conector soportado → conexión UC
# autorizada → tablas de origen → catálogo/schema destino → frecuencia → primera carga.
# Verifica filas y progreso, añade un cambio en origen y comprueba la siguiente carga.
# Conector estándar JDBC, solo con URL/driver/secret scope reales:
# source = (spark.read.format("jdbc").option("url", jdbc_url)
#   .option("dbtable", "dbo.customers").option("user", user)
#   .option("password", dbutils.secrets.get("practice", "jdbc-password")).load())
# source.write.format("delta").mode("append").saveAsTable("jdbc_bronze")
# ODBC suele servir clientes que consultan Databricks; no es un formato spark.read universal.

## PARTE 5 - PREGUNTAS TIPO EXAMEN
Preguntas originales de práctica; no son preguntas oficiales.

### Pregunta 1
COPY INTO omite un archivo ya cargado. ¿Qué evita con ello?

A. Todos los duplicados por clave

B. Releer ese archivo normalmente

C. Los valores nulos

D. Errores del modelo de negocio

### Pregunta 2
¿Qué elección reduce desarrollo de ingesta para una fuente empresarial soportada?

A. Un CSV manual permanente

B. collect

C. Lakeflow Connect managed

D. DROP TABLE

### Pregunta 3
¿Qué debes gestionar con una API REST propia?

A. Paginación, cursor, reintentos y autenticación

B. Solo el color del notebook

C. Siempre un stream Kafka

D. Nada: Spark lo infiere todo

### Respuestas y explicación
**1. B** — El seguimiento de archivos no implica deduplicación de entidades.

**2. C** — El servicio gestiona parte del ciclo de ingesta.

**3. A** — Son responsabilidades de la extracción personalizada.

### Documentación oficial
- [Lakeflow Connect](https://docs.databricks.com/aws/en/ingestion/lakeflow-connect/)
- [COPY INTO](https://docs.databricks.com/aws/en/sql/language-manual/delta-copy-into)

## PARTE 6 - RETO FINAL
Compara una carga COPY INTO con tu cliente ficticio paginado. Añade un replay, demuestra sus efectos y diseña la estrategia de idempotencia.

Anota tu decisión, implementa el código y muestra evidencias. No se incluye solución para este reto.

In [ ]:
# TU RETO: código y verificaciones
